# E1 · Re-evaluate the published adapters on the gold set

**Goal.** Reproduce the thesis numbers from the **public** QLoRA adapters
([`AntoniusPerf/runic-ocr-qwen-vl-lora`](https://huggingface.co/AntoniusPerf/runic-ocr-qwen-vl-lora)) and add what the thesis lacked:

* **inscription-level 95 % CIs** (113 lines come from only 28 inscriptions);
* **paired significance tests** between models;
* full **error analysis** (substitutions / deletions / insertions, confusion matrix, top confusions);
* **5-best beam hypotheses** per line, the input for decoding experiments (E5: lexicon / character-LM re-ranking).

**Runtime:** ≈ 1–1.5 h on one T4 for three models.

### How to run (fire and forget)
1. Kaggle → **New Notebook** → File → **Import Notebook** → this `.ipynb`.
2. Right panel: **Accelerator → GPU T4 x2 (or P100)**, **Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (Hugging Face token with *write* access). It is needed only to mirror
   results to the private dataset `AntoniusPerf/runic-ocr-results`; without it, results stay in the Output tab.
4. **Add Input** (optional, recommended): your dataset `zhopa228/val-dataset`. It has the one gold image missing from the
   repo copy, so evaluation covers 113 lines instead of 112.
5. **Save Version → Save & Run All (Commit)**, then close the tab. When the run finishes, open **Output**: everything is in
   `results/E1_reeval/`, with the summary in `REPORT.md`.

If the session stops early (time limit, GPU error), just run it again: finished steps are skipped (and pulled back from the HF dataset if the Output was lost).

In [ ]:
# ── Configuration (the only cell you may want to edit) ──
REPO_URL = "https://github.com/kekys778/RUNIC-OCR.git"
REPO_REF = "main"  # branch or tag of the code to run

ADAPTER_REPO = "AntoniusPerf/runic-ocr-qwen-vl-lora"  # public QLoRA adapters from the thesis
MODEL_KEYS = [
    "qwen25vl-7b",
    "qwen3vl-8b",
    "qwen3vl-2b",
]  # registry keys in src/runic_ocr_experiments.py
N_BEST = 5  # beam hypotheses saved per line (0 or 1 = greedy only)
RESULTS_HF_REPO = (
    "AntoniusPerf/runic-ocr-results"  # private HF dataset for results ("" = local only)
)
SEED = 0

In [2]:
# ── Bootstrap: dependencies, code, sanity checks (≈2 min) ── no need to edit ──
import subprocess
import sys
import time
from pathlib import Path

T0 = time.time()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORK / "results"
REPO = Path("/tmp/RUNIC-OCR")  # outside /kaggle/working, so it is not saved as output

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "bitsandbytes>=0.46.1",
        "qwen-vl-utils",
        "rapidfuzz",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "tabulate"], check=True)
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)],
        check=True,
    )
sys.path.insert(0, str(REPO / "src"))

import torch  # noqa: E402

from runic_kaggle import ResultSink, environment_report, find_adapter, get_secret, load_gold  # noqa: E402

assert torch.cuda.is_available(), (
    "No GPU. Kaggle: Settings → Accelerator → GPU T4 x2 (or P100), then run again."
)
HF_TOKEN = get_secret("HF_TOKEN")
if not HF_TOKEN:
    print(
        "⚠ No HF_TOKEN secret: the public adapters still download, but results are only saved locally."
    )
ENV = environment_report()
print(ENV)

⚠ No HF_TOKEN secret: the public adapters still download, but results are only saved locally.
{'python': '3.12.13', 'platform': 'Linux-6.12.90+-x86_64-with-glibc2.35', 'torch': '2.10.0+cu128', 'transformers': '5.0.0', 'peft': '0.19.1', 'bitsandbytes': '0.50.2', 'accelerate': '1.13.0', 'cuda': True, 'gpus': ['Tesla T4', 'Tesla T4']}


## 1. Gold set, adapters, result folder

In [3]:
from huggingface_hub import snapshot_download

import runic_ocr_experiments as H
import runic_reports as R
from runic_vlm import free_gpu, load_qwen, predict

gold = load_gold(image_roots=["/kaggle/input"])
print(f"gold set: {len(gold)} lines, {gold['inscription'].nunique()} inscriptions")

ADAPTERS = Path(snapshot_download(ADAPTER_REPO, token=HF_TOKEN))
sink = ResultSink("E1_reeval", OUT_DIR, hf_repo=RESULTS_HF_REPO, hf_token=HF_TOKEN)
sink.save_json(
    "environment.json",
    {**ENV, "repo_ref": REPO_REF, "adapter_repo": ADAPTER_REPO, "n_gold": len(gold)},
)
print("already done:", sorted(sink.done_steps()))

[gold] no image for 1 line(s), skipped: ['5810_0_karþi-kuml-þusi-eftiR-urmar.png']
gold set: 112 lines, 28 inscriptions
already done: []


## 2. Predict with each adapter
Greedy decoding gives the thesis-protocol prediction; a separate beam search stores the top-5 hypotheses.

In [4]:
for key in MODEL_KEYS:
    step = f"predict_{key}"
    if sink.is_done(step):
        print(f"✓ {key}: done earlier")
        continue
    try:
        adapter = find_adapter(ADAPTERS, H.EXPERIMENTS[key].model_id)
    except FileNotFoundError as e:
        print(f"⚠ {key}: {e}")
        sink.mark_done(step, status="adapter_missing")
        continue
    t = time.time()
    print(f"\n▶ {key}  ({adapter})")
    model, proc, cfg = load_qwen(key, str(adapter))
    recs = predict(model, proc, cfg, gold["path"].tolist(), n_best=N_BEST)
    del model
    free_gpu()
    preds = R.predictions_frame(gold, recs)
    sink.save_csv(f"{key}_gold_predictions.csv", preds)
    s = R.score(preds, SEED)
    print(
        f"  CER {s['CER']:.2f} %  (inscription CI {s['ci_inscription'][0]:.1f}–{s['ci_inscription'][1]:.1f})"
    )
    sink.mark_done(step, CER=round(s["CER"], 2), minutes=round((time.time() - t) / 60, 1))

▶ qwen25vl-7b  (/root/.cache/huggingface/hub/models--AntoniusPerf--runic-ocr-qwen-vl-lora/snapshots/a1fc6371387f904e5a50d0e89100c7f6a383b092/qwen25vl-7b)
  25/112 images, 2.9 s/img
  50/112 images, 2.6 s/img
  75/112 images, 2.4 s/img
  100/112 images, 2.4 s/img
  CER 55.40 %  (inscription CI 43.7–70.5)
▶ qwen3vl-8b  (/root/.cache/huggingface/hub/models--AntoniusPerf--runic-ocr-qwen-vl-lora/snapshots/a1fc6371387f904e5a50d0e89100c7f6a383b092/qwen3vl-8b)
  25/112 images, 4.8 s/img
  50/112 images, 3.8 s/img
  75/112 images, 3.5 s/img
  100/112 images, 3.4 s/img
  CER 65.23 %  (inscription CI 54.0–78.7)
▶ qwen3vl-2b  (/root/.cache/huggingface/hub/models--AntoniusPerf--runic-ocr-qwen-vl-lora/snapshots/a1fc6371387f904e5a50d0e89100c7f6a383b092/qwen3vl-2b)
  25/112 images, 2.9 s/img
  50/112 images, 2.1 s/img
  75/112 images, 1.9 s/img
  100/112 images, 1.9 s/img
  CER 73.76 %  (inscription CI 68.5–81.1)


## 3. Summary: reproduction check, significance, error analysis

In [5]:
import itertools

import pandas as pd

preds = {
    k: pd.read_csv(sink.path(f"{k}_gold_predictions.csv"))
    for k in MODEL_KEYS
    if sink.path(f"{k}_gold_predictions.csv").exists()
}
thesis = {}
for k in preds:  # thesis predictions shipped in the repo, if any
    f = REPO / "results" / "predictions" / f"{k}_finetuned_gold.csv"
    if f.exists():
        thesis[k] = pd.read_csv(f)

table = R.models_table(preds, thesis=thesis, seed=SEED)
sink.save_csv("models.csv", table)

pairs = []
for a, b in itertools.combinations(table["model"], 2):
    c = R.compare(preds[a], preds[b], seed=SEED)
    pairs.append(
        {
            "A": a,
            "B": b,
            "delta_CER_A_minus_B": round(c["delta"], 2),
            "CI95": f"{c['ci_low']:.1f}…{c['ci_high']:.1f}",
            "significant": c["significant"],
        }
    )
pairs = pd.DataFrame(pairs)
sink.save_csv("pairwise.csv", pairs)

for k in preds:  # error-analysis report + figures per model
    subprocess.run(
        [
            sys.executable,
            str(REPO / "scripts" / "analyze_predictions.py"),
            "--pred",
            str(sink.path(f"{k}_gold_predictions.csv")),
            "--baseline",
            "",
            "--out",
            str(sink.path(f"analysis_{k}")),
        ],
        check=True,
    )

report = f"""# E1 · Re-evaluation of the published adapters

Run finished {time.strftime("%Y-%m-%d %H:%M")} · {len(gold)} gold lines / {gold["inscription"].nunique()} inscriptions ·
code `{REPO_REF}` · GPU {ENV.get("gpus")} · {(time.time() - T0) / 3600:.1f} h.

## Models
`thesis_CER_same_lines` is the thesis CER recomputed on exactly these lines. A gap of more than ~1 pp means
the environment does not reproduce the thesis run.

{R.df_to_md(table)}

## Pairwise differences (paired bootstrap over inscriptions)

{R.df_to_md(pairs) if len(pairs) else "(fewer than two models)"}

Per-model error analysis: `analysis_<model>/REPORT.md`. Beam hypotheses: column `nbest` in `<model>_gold_predictions.csv`.
"""
sink.save_text("REPORT.md", report)
sink.mark_done("summary")
print(report)

saved /kaggle/working/results/E1_reeval/analysis_qwen25vl-7b
{"CER": 55.4, "ci95_line_bootstrap": [49.0, 62.2], "ci95_inscription_bootstrap": [43.7, 70.5], "errors": {"S": 137, "D": 362, "I": 14}}
saved /kaggle/working/results/E1_reeval/analysis_qwen3vl-8b
{"CER": 65.23, "ci95_line_bootstrap": [59.5, 71.1], "ci95_inscription_bootstrap": [54.0, 78.7], "errors": {"S": 328, "D": 232, "I": 44}}
saved /kaggle/working/results/E1_reeval/analysis_qwen3vl-2b
{"CER": 73.76, "ci95_line_bootstrap": [70.0, 77.7], "ci95_inscription_bootstrap": [68.5, 81.1], "errors": {"S": 334, "D": 322, "I": 27}}
# E1 · Re-evaluation of the published adapters
Run finished 2026-10-01 15:22 · 112 gold lines / 28 inscriptions ·
code `main` · GPU ['Tesla T4', 'Tesla T4'] · 0.4 h.
## Models
`thesis_CER_same_lines` is the thesis CER recomputed on exactly these lines. A gap of more than ~1 pp means
the environment does not reproduce the thesis run.
| model | lines | CER | thesis_CER_same_lines | CI95_line | CI95_inscripti

## Results (Kaggle run, 2026-10-01, 2× T4, 0.4 h)

| model | CER, % | thesis CER, % | 95 % CI (inscriptions) | substitutions / deletions / insertions |
|---|---:|---:|---:|---|
| **Qwen2.5-VL-7B** | **55.40** | 55.40 on the same 112 lines (54.27 on 113) | 43.7–70.5 | 137 / 362 / 14 |
| Qwen3-VL-8B | 65.23 | 67.86 (113 lines) | 54.0–78.7 | 328 / 232 / 44 |
| Qwen3-VL-2B | 73.76 | 73.55 (113 lines) | 68.5–81.1 | 334 / 322 / 27 |

* **The thesis reproduces.** The public Qwen2.5-VL-7B adapter gives exactly the thesis CER on the same lines, in a newer
  environment (transformers 5.0, peft 0.19). The Qwen3 models are within about 2.6 pp of the thesis, which also used
  one more gold line.
* **The model ranking is real.** All three pairwise differences stay significant under the stricter inscription-level
  paired bootstrap: 7B vs. 8B −9.8 pp (CI −14.3…−3.9), 7B vs. 2B −18.4 pp, 8B vs. 2B −8.5 pp.
* **The models fail differently.** Qwen2.5-VL-7B mostly *omits* characters (71 % of its errors are deletions), while both
  Qwen3-VL models mostly *misread* them (49–54 % substitutions) and insert more. The best model is conservative: it
  writes less rather than guessing.
* Only 112 of 113 lines were evaluated because `zhopa228/val-dataset` was not attached. Each model's 5-best beam
  hypotheses are in the run's Output (`results/E1_reeval/<model>_gold_predictions.csv`, column `nbest`).